In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip -q /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs_split.zip

In [3]:
import os
len(os.listdir("/content/content/OMR_5Fold_ROIs_split/Fold_1/train/crossedout"))

1500

In [ ]:
!rm -rf checkpoints/
!rm -rf GAN_Images/
!rm -rf img/

In [4]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import glob
from tqdm import tqdm

# ==========================================
# 1. CẤU HÌNH SIÊU THAM SỐ (HYPERPARAMETERS)
# ==========================================
BATCH_SIZE = 32          # Chống overfit cho dữ liệu nhỏ
IMAGE_SIZE = 64          # Kích thước chuẩn OMR ROI
Z_DIM = 100              # Chiều dài vector nhiễu
NUM_EPOCHS = 300         # Số vòng lặp hội tụ
LR_G = 0.0002            # Learning Rate cho Generator
LR_D = 0.00005            # TTUR: Kìm hãm Discriminator
BETA1 = 0.5              # Tham số Adam optimizer chuẩn của DCGAN
NUM_GENERATE = 1000      # Số ảnh cần đẻ ra mỗi Fold

# Đường dẫn
K_FOLDS_DIR = "/content/content/OMR_5Fold_ROIs_split"
GAN_IMG_DIR = "/content/drive/MyDrive/OMR-Datasets/gan/GAN_Images"
IMG_EPOCH_DIR = "/content/drive/MyDrive/OMR-Datasets/gan/img"
CHECKPOINT_DIR = "/content/drive/MyDrive/OMR-Datasets/gan/checkpoints"

# Thiết bị
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🔥 Đang chạy trên thiết bị: {DEVICE}")

# ==========================================
# 2. KHỞI TẠO KIẾN TRÚC MẠNG DCGAN
# ==========================================
# Hàm khởi tạo trọng số ngẫu nhiên chuẩn phân phối Normal
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Mạng sinh (Generator)
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(Z_DIM, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh() # Đưa pixel về dải [-1, 1]
        )
    def forward(self, input):
        return self.main(input)

# Mạng phân biệt (Discriminator)
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3), # thêm dropout
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3), # thêm dropout
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3), # thêm dropout
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid() # Đưa xác suất về [0, 1]
        )
    def forward(self, input):
        return self.main(input)

# ==========================================
# 3. QUẢN LÝ DỮ LIỆU TỰ ĐỘNG CHO TỪNG FOLD
# ==========================================
class OMRDataset(Dataset):
    def __init__(self, folder_path, transform):
        self.image_paths = glob.glob(os.path.join(folder_path, "*.*"))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.transform(img)

# Transform chuẩn cho DCGAN (Resize và chuẩn hóa về [-1, 1])

class SquarePad:
    def __call__(self, image):
        w, h = image.size
        max_wh = np.max([w, h])
        hp = int((max_wh - w) / 2)
        vp = int((max_wh - h) / 2)
        padding = (hp, vp, hp, vp)
        return F.pad(image, padding, (255, 255, 255), 'constant')

transform = transforms.Compose([
    SquarePad(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# ==========================================
# 4. VÒNG LẶP HUẤN LUYỆN 5 FOLDS (ZERO-LEAKAGE)
# ==========================================
for fold in range(1, 6):
    print(f"\n{'='*50}")
    print(f"🚀 BẮT ĐẦU HUẤN LUYỆN DCGAN CHO FOLD {fold}")
    print(f"{'='*50}")

    # Đường dẫn lấy 1500 ảnh augmentation
    train_dir = os.path.join(K_FOLDS_DIR, f"Fold_{fold}", "train", "crossedout")

    # Thư mục lưu ảnh đánh giá theo epoch
    progress_dir = os.path.join(IMG_EPOCH_DIR, f"Fold_{fold}")
    os.makedirs(progress_dir, exist_ok=True)

    #Thư mục lưu checkpoint
    checkpoint_dir = os.path.join(CHECKPOINT_DIR, f"Fold_{fold}")
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Nơi lưu 1000 ảnh GAN mới đẻ ra
    gan_output_dir = os.path.join(GAN_IMG_DIR, f"Fold_{fold}")
    os.makedirs(gan_output_dir, exist_ok=True)

    # Load Dữ liệu
    dataset = OMRDataset(train_dir, transform)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    print(f"📁 Số lượng ảnh dùng để dạy GAN ở Fold {fold}: {len(dataset)}")

    # KHỞI TẠO MẠNG MỚI HOÀN TOÀN CHO FOLD NÀY (Để không mang "trí nhớ" từ Fold trước sang)
    netG = Generator().to(DEVICE)
    netG.apply(weights_init)

    netD = Discriminator().to(DEVICE)
    netD.apply(weights_init)

    # Hàm Loss và Optimizer (TTUR)
    criterion = nn.BCELoss()
    optimizerD = optim.Adam(netD.parameters(), lr=LR_D, betas=(BETA1, 0.999))
    optimizerG = optim.Adam(netG.parameters(), lr=LR_G, betas=(BETA1, 0.999))

    # Vector nhiễu cố định để theo dõi cùng 1 gốc sinh ảnh qua thời gian
    fixed_noise = torch.randn(32, Z_DIM, 1, 1, device=DEVICE)

    # Vòng lặp Epochs
    for epoch in range(NUM_EPOCHS):
        for i, data in enumerate(dataloader):
            real_images = data.to(DEVICE)
            b_size = real_images.size(0)

            # --- 1. TRAIN DISCRIMINATOR ---
            netD.zero_grad()
            label_real = torch.full((b_size,), 0.9, dtype=torch.float, device=DEVICE)
            output_real = netD(real_images).view(-1)
            errD_real = criterion(output_real, label_real)
            errD_real.backward()

            noise = torch.randn(b_size, Z_DIM, 1, 1, device=DEVICE)
            fake_images = netG(noise)
            label_fake = torch.full((b_size,), 0.0, dtype=torch.float, device=DEVICE)
            output_fake = netD(fake_images.detach()).view(-1)
            errD_fake = criterion(output_fake, label_fake)
            errD_fake.backward()
            optimizerD.step()

            # --- 2. TRAIN GENERATOR ---
            netG.zero_grad()
            label_real_for_G = torch.full((b_size,), 0.9, dtype=torch.float, device=DEVICE)
            output_fake_for_G = netD(fake_images).view(-1)
            errG = criterion(output_fake_for_G, label_real_for_G)
            errG.backward()
            optimizerG.step()

        # In tiến trình mỗi 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"   [Fold {fold}] Epoch [{epoch+1}/{NUM_EPOCHS}] | Loss D: {errD_real.item()+errD_fake.item():.4f} | Loss G: {errG.item():.4f}")

        if (epoch+1) % 10 == 0:
            #lưu ảnh check
            with torch.no_grad():
                fake = netG(fixed_noise).detach().cpu()
                vutils.save_image(fake, f"{progress_dir}/epoch_{epoch+1}.png", normalize=True)
            #lưu trọng số
            torch.save(netG.state_dict(), f"{checkpoint_dir}/netG_epoch_{epoch+1}.pth")
            torch.save(netD.state_dict(), f"{checkpoint_dir}/netD_epoch_{epoch+1}.pth")
            print(f"-> Đã checkpoint tại epoch {epoch+1}")

        torch.save(netG.state_dict(), f"{checkpoint_dir}/netG_last.pth")
        torch.save(netD.state_dict(), f"{checkpoint_dir}/netD_last.pth")
    # ==========================================
    # 5. SINH ẢNH VÀ LƯU VÀO KỊCH BẢN 2 (WITH GAN)
    # ==========================================
    print(f"🎨 Đang sinh {NUM_GENERATE} ảnh giả cho Fold {fold}...")
    netG.eval() # Tắt tính năng train để sinh ảnh

    with torch.no_grad():
        # Sinh từng đợt để không bị tràn RAM GPU
        batch_generate = 100
        for b in range(NUM_GENERATE // batch_generate):
            noise = torch.randn(batch_generate, Z_DIM, 1, 1, device=DEVICE)
            fake_imgs = netG(noise)

            # Chuyển đổi tensor dải [-1, 1] về ảnh RGB dải [0, 1]
            fake_imgs = (fake_imgs + 1) / 2.0

            # Lưu ra ổ đĩa
            for j in range(batch_generate):
                img_idx = b * batch_generate + j
                save_path = os.path.join(gan_output_dir, f"gan_fold{fold}_{img_idx}.jpg")
                vutils.save_image(fake_imgs[j], save_path)
    # Đánh giá fid


    print(f"✅ Đã lưu {NUM_GENERATE} ảnh GAN vào thư mục kịch bản 2 của Fold {fold}.\n")

print("🎉 XUẤT SẮC! ĐÃ HOÀN TẤT HUẤN LUYỆN VÀ SINH ẢNH ZERO-LEAKAGE CHO TOÀN BỘ 5 FOLDS!")

🔥 Đang chạy trên thiết bị: cuda

🚀 BẮT ĐẦU HUẤN LUYỆN DCGAN CHO FOLD 1
📁 Số lượng ảnh dùng để dạy GAN ở Fold 1: 1500
   [Fold 1] Epoch [10/300] | Loss D: 1.1876 | Loss G: 1.4030
-> Đã checkpoint tại epoch 10
   [Fold 1] Epoch [20/300] | Loss D: 1.0988 | Loss G: 1.3974
-> Đã checkpoint tại epoch 20
   [Fold 1] Epoch [30/300] | Loss D: 1.2363 | Loss G: 1.0388
-> Đã checkpoint tại epoch 30
   [Fold 1] Epoch [40/300] | Loss D: 0.9116 | Loss G: 1.6569
-> Đã checkpoint tại epoch 40
   [Fold 1] Epoch [50/300] | Loss D: 0.8321 | Loss G: 1.6835
-> Đã checkpoint tại epoch 50
   [Fold 1] Epoch [60/300] | Loss D: 0.9251 | Loss G: 1.7557
-> Đã checkpoint tại epoch 60
   [Fold 1] Epoch [70/300] | Loss D: 0.7931 | Loss G: 2.1417
-> Đã checkpoint tại epoch 70
   [Fold 1] Epoch [80/300] | Loss D: 0.8120 | Loss G: 2.0752
-> Đã checkpoint tại epoch 80
   [Fold 1] Epoch [90/300] | Loss D: 0.8360 | Loss G: 1.5908
-> Đã checkpoint tại epoch 90
   [Fold 1] Epoch [100/300] | Loss D: 0.7712 | Loss G: 2.0096
->

In [6]:
!pip install pytorch-fid

In [10]:
import os
import shutil
from PIL import Image, ImageOps

# ==========================================
# 0. ĐỊNH NGHĨA HÀM ĐẮP VIỀN BẰNG PIL
# ==========================================
def pil_square_pad(img, fill_color=(255, 255, 255)):
    """Đắp viền trắng đều 2 bên để ảnh chữ nhật thành ảnh vuông"""
    w, h = img.size
    max_wh = max(w, h)

    # Tính toán lề trái, trên, phải, dưới
    left = int((max_wh - w) / 2)
    top = int((max_wh - h) / 2)
    right = max_wh - w - left
    bottom = max_wh - h - top

    padding = (left, top, right, bottom)
    return ImageOps.expand(img, padding, fill=fill_color)

# ==========================================
# 1. KHAI BÁO ĐƯỜNG DẪN
# ==========================================
path_to_real_original = "/content/content/OMR_5Fold_ROIs_split/Fold_1/train/crossedout" # Thư mục chứa ảnh thật gốc
path_to_real_resized = "/content/Crossed_out_Real_64x64"
path_to_fake = "/content/drive/MyDrive/OMR-Datasets/gan/GAN_Images/Fold_2" # Thư mục chứa ảnh GAN

# 2. Làm sạch thư mục tạm
if os.path.exists(path_to_real_resized):
    shutil.rmtree(path_to_real_resized)
os.makedirs(path_to_real_resized)

# ==========================================
# 3. TIỀN XỬ LÝ ẢNH THẬT ĐÚNG CHUẨN (PAD -> RESIZE)
# ==========================================
print("Đang đắp viền và đồng bộ kích thước ảnh thật về 64x64...")
valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp')

for img_name in os.listdir(path_to_real_original):
    if img_name.lower().endswith(valid_extensions):
        img_path = os.path.join(path_to_real_original, img_name)
        try:
            # 1. Đọc ảnh
            img = Image.open(img_path).convert('RGB')

            # 2. Đắp viền trắng thành hình vuông TRƯỚC
            img_padded = pil_square_pad(img)

            # 3. Mới Resize về 64x64
            img_resized = img_padded.resize((64, 64), Image.Resampling.BICUBIC)

            # 4. Lưu lại
            img_resized.save(os.path.join(path_to_real_resized, img_name))
        except Exception as e:
            print(f"Lỗi khi xử lý ảnh {img_name}: {e}")

print(f"✅ Đã xử lý xong! Dữ liệu mượt mà lưu tại: {path_to_real_resized}")

# ==========================================
# 4. TÍNH TOÁN FID
# ==========================================
print("\n🚀 Đang tính toán lại chỉ số FID (Apples-to-Apples)...")
!python -m pytorch_fid {path_to_real_resized} {path_to_fake} --device cuda:0

Đang đắp viền và đồng bộ kích thước ảnh thật về 64x64...
✅ Đã xử lý xong! Dữ liệu mượt mà lưu tại: /content/Crossed_out_Real_64x64

🚀 Đang tính toán lại chỉ số FID (Apples-to-Apples)...
100% 30/30 [00:06<00:00,  4.50it/s]
100% 20/20 [00:09<00:00,  2.15it/s]
FID:  98.02178242361663
